# NB01 · Semana 2 — Recorrido del stack de datos
**Ingeniería de Datos — LCD · UCA Rosario · 2026** · grupo06 · Clima de Sídney (Open-Meteo)

Cada pieza del stack cubre una etapa del ciclo de vida. En el servidor de la cátedra:

| Componente | Etapa | Dónde |
|---|---|---|
| MinIO | Almacenamiento (data lake, S3) | bucket `grupo06` |
| DuckDB | Transformación (motor in-process de dbt) | `dbt/warehouse.duckdb` |
| Airflow | Orquestación | https://airflow.nexoacademia.org |
| PostgreSQL | Serving (capa gold) | base `warehouse`, schema `grupo06` |
| Metabase | BI | https://metabase.nexoacademia.org |

In [ ]:
import os, sys
from pathlib import Path
# Raíz del proyecto = carpeta padre de notebooks/ (en el servidor: ~/work/dags)
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
import entorno
entorno.info()

## 1. MinIO: el lake
Nos conectamos por protocolo S3 con las credenciales del grupo (variables de entorno, nunca escritas en el código). Solo tenemos permiso sobre **nuestro** bucket: listar otros buckets da 403, y eso es aislamiento entre grupos, no un error.

In [ ]:
import boto3
so = entorno.STORAGE_OPTIONS
s3 = boto3.client("s3", endpoint_url=so["AWS_ENDPOINT_URL"], aws_access_key_id=so["AWS_ACCESS_KEY_ID"],
                  aws_secret_access_key=so["AWS_SECRET_ACCESS_KEY"], region_name="us-east-1")
r = s3.list_objects_v2(Bucket=entorno.BUCKET, Prefix="bronze/clima_sidney/", MaxKeys=20)
for o in r.get("Contents", []):
    print(f"{o['Size']:>9}  {o['Key']}")

## 2. DuckDB: motor analítico in-process
No es un servidor: corre dentro de este proceso. dbt lo usa para transformar.

In [ ]:
import duckdb
print("DuckDB", duckdb.__version__)
duckdb.sql("select 42 as respuesta, current_date as hoy").show()

## 3. PostgreSQL: warehouse de serving
Acá publicamos la capa gold para que Metabase la consulte.

In [ ]:
from sqlalchemy import create_engine, text
with create_engine(entorno.pg_url()).connect() as con:
    print(con.execute(text("select version()")).scalar())
    print(con.execute(text("select current_schema()")).scalar())